In [1]:
import os, re, hashlib
from pathlib import Path
import pprint
from dotenv import load_dotenv
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

C:\Users\Sadha\AppData\Local\Temp\ipykernel_2508\3421141935.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [2]:
# ---- Cell A: setup ----
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [3]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [4]:
# Create Project root
PROJECT_ROOT = Path(r"C:\NorthStar-rag")
load_dotenv(PROJECT_ROOT / ".env")# OPENAI_API_KEY, EMBEDDING_MODEL

True

In [5]:
# Define constant
DATA_DIR    = PROJECT_ROOT / "data" / "Resources_gemini"
PERSIST_DIR = PROJECT_ROOT / "vector_db" / "chroma"
COLLECTION  = "northstar_docs"
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

In [6]:
# ---- cell 10: load PDFs -> Markdown -> chunks ----
FOOTER_RE = re.compile(r"^(?:.*\(NSB-[A-Z]+-\d{4}-V\d+\).*|\s*Page \d+ of \d+\s*)$\n?", re.M)

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "title"), ("##", "section"), ("###", "subsection")],
    strip_headers=False,
)
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "],
)

In [7]:
md_text = pymupdf4llm.to_markdown(str(r"C:\NorthStar-rag\data\Resources_gemini\NorthStar_Bank_Credit_Card_Customer_Guide.pdf"), show_progress=False)



=== Document parser messages ===
Using Tesseract for OCR processing.


In [8]:
#Load the document and spit it
def load_pdf_chunks(pdf_path: Path):
    md_text = pymupdf4llm.to_markdown(str(pdf_path), show_progress=False)
    #print(md_text)
    md_text = FOOTER_RE.sub("", md_text)
    #print(md_text)
    sections = header_splitter.split_text(md_text)
    chunks = char_splitter.split_documents(sections)
    for i, c in enumerate(chunks):
        c.metadata.update(source=pdf_path.name, chunk_index=i,
                          has_table="|---|" in c.page_content)
    return chunks


In [9]:
all_chunks = []
for pdf in sorted(DATA_DIR.glob("*.pdf")):
    chunks = load_pdf_chunks(pdf)
    print(chunks)
    print(f"{pdf.name}: {len(chunks)} chunks")
    all_chunks.extend(chunks)
print("total chunks:", len(all_chunks))


=== Document parser messages ===
Using Tesseract for OCR processing.
[Document(metadata={'source': 'NorthStar_Bank_Credit_Card_Customer_Guide.pdf', 'chunk_index': 0, 'has_table': False}, page_content='NORTHSTAR BANK, N.A. • CARD SERVICES DIVISION'), Document(metadata={'title': 'Credit Card Customer Guide', 'source': 'NorthStar_Bank_Credit_Card_Customer_Guide.pdf', 'chunk_index': 1, 'has_table': False}, page_content='# Credit Card Customer Guide  \nEffective Date: November 1, 2026  |  Document ID: NSB-CC-2026-V7  |  Member FDIC • Equal Opportunity Lender'), Document(metadata={'title': 'Credit Card Customer Guide', 'section': '1. Introduction & Welcome to NorthStar Bank Credit Cards', 'source': 'NorthStar_Bank_Credit_Card_Customer_Guide.pdf', 'chunk_index': 2, 'has_table': False}, page_content='## 1. Introduction & Welcome to NorthStar Bank Credit Cards  \nWelcome to NorthStar Bank Credit Cards. Whether you are earning everyday cash back on groceries and gas, collecting flexible travel 

In [10]:
# ---- create the persistent Chroma store and store the embeddings ----
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),        # saved to C:\NorthStar-rag\vector_db\chroma
    collection_metadata={"hnsw:space": "cosine"},
)

ids = [hashlib.md5(f"{c.metadata['source']}|{c.metadata['chunk_index']}|{c.page_content}".encode()).hexdigest()
       for c in all_chunks]

vector_store.reset_collection()                # wipe and rebuild
vector_store.add_documents(all_chunks, ids=ids)  # embeds with OpenAI and writes to disk
print("vectors stored:", vector_store._collection.count())

vectors stored: 177


In [11]:
vector_store = Chroma(collection_name=COLLECTION, embedding_function=embeddings,
                      persist_directory=str(PERSIST_DIR))
print("vectors on disk:", vector_store._collection.count())   # -> 177

for doc, score in vector_store.similarity_search_with_score(
        "What is the foreign transaction fee on the Travel Horizon card?", k=3):
    print(f"{score:.3f} | {doc.metadata['source']} | {doc.metadata.get('section')}")


vectors on disk: 177
0.437 | NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 8. Frequently Asked Questions (52 Comprehensive FAQs)
0.468 | NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 8. Frequently Asked Questions (52 Comprehensive FAQs)
0.489 | NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 5. Fees, Rates & Charges Schedule


In [12]:

data = vector_store._collection.get(include=["embeddings", "documents", "metadatas"], limit=3)
print(len(data["embeddings"][0]))   # 1536 dimensions for text-embedding-3-small
data["embeddings"][:3]


1536


array([[-0.03451538, -0.03695679,  0.02410889, ...,  0.00636292,
         0.01808167,  0.00850677],
       [ 0.00747299, -0.04064941,  0.00317955, ...,  0.02453613,
        -0.02012634, -0.02642822],
       [-0.05142212, -0.00938415,  0.00942993, ..., -0.00780106,
         0.00662613, -0.00089741]], shape=(3, 1536))

In [13]:
retriever = BM25Retriever.from_documents(all_chunks,k=2)
query = "What is the foreign transaction fee on the Travel Horizon card?"
result1 = retriever.invoke(query)
print(query)
for i, doc in enumerate(result1, start=1):
    print(
        f"--- Match {i} | Source: {doc.metadata.get('source', 'Unknown')} ---"
    )
    print(doc.page_content[:200])
    print()

What is the foreign transaction fee on the Travel Horizon card?
--- Match 1 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### 33. What travel insurance benefits come with the Travel Horizon Visa Signature®?  
When you pay for your common-carrier ticket with your Travel Horizon card, you receive Trip Cancellation/Interrup

--- Match 2 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### 32. Should I pay in local currency or U.S. Dollars when abroad?  
Always choose to pay in the local foreign currency. Selecting U.S. Dollars triggers Dynamic Currency Conversion (DCC) by the forei



## Retrival

In [14]:
def build_retriever(source=None, k=20):
    docs = [
        c for c in all_chunks
        if source is None or c.metadata.get("source") == source
    ]

    bm25 = BM25Retriever.from_documents(docs)
    bm25.k = k

    dense = vector_store.as_retriever(
        search_kwargs={"k": k}
    )

    return EnsembleRetriever(
        retrievers=[bm25, dense],
        weights=[0.4, 0.6]
    )

In [15]:
hybrid = build_retriever(
    "NorthStar_Bank_Credit_Card_Customer_Guide.pdf"
)
query = "Does NorthStar Bank have a late payment grace period?"
candidates = hybrid.invoke(query)
for i, doc in enumerate(candidates, start=1):
    print(
        f"--- Match {i} | Source: {doc.metadata.get('source', 'Unknown')} ---"
    )
    print(doc.page_content[:200])
    print()

--- Match 1 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### 19. Does NorthStar Bank have a late payment grace period?  
No. Unlike personal loans, credit cards do not have a post-due-date grace period for late fees. Payments received after 5:00 PM EST on y

--- Match 2 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### 46. Does NorthStar Bank offer hardship programs if I lose my job or face a medical emergency?  
Yes. Call our Cardmember Assistance Team at 1-800-555-0140 to discuss reduced fixed-APR payment plan

--- Match 3 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### 5. Can NorthStar Bank automatically increase my credit limit?  
Yes. NorthStar conducts periodic account reviews every 6 to 12 months. Cardholders who maintain low utilization, on-time payments, a

--- Match 4 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### C. Requesting a Balance Transfer  
1. Verify Eligibility: Balance transfers cannot be used to pay off 

In [16]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-base"
)

print("Loaded successfully")

c:\NorthStar-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 276.34it/s]


Loaded successfully


In [17]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

reranker = CrossEncoderReranker(
    model=model,
    top_n=4
)

print("Reranker created")

Reranker created


In [18]:
from langchain_classic.retrievers import ContextualCompressionRetriever

retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid
)

print("Compression retriever created")

Compression retriever created


In [19]:
queries = [
    "How much is the late payment fee?",
    "What is the foreign transaction fee on the Travel Horizon card?",
    "What APR applies to cash advances?",
    "Is there an annual fee for the card?",
    "What happens if I miss a payment?",
    "How is interest calculated on purchases?",
    "What is the penalty APR?",
    "What are the rewards benefits of the Travel Horizon card?",
    "How can I avoid paying interest?",
    "What is the grace period for purchases?"
]

for query in queries:
    print("\n" + "="*100)
    print("QUERY:", query)
    print("="*100)

    results = retriever.invoke(query)

    for i, doc in enumerate(results, start=1):
        print(f"\n--- Match {i} | Source: {doc.metadata.get('source')} ---")
        print(doc.page_content[:500])


QUERY: How much is the late payment fee?

--- Match 1 | Source: NorthStar_Bank_Personal_Loan_Customer_Handbook.pdf ---
### 26. What happens if I miss a payment or pay after the grace period?  
On day 11 past your due date, a Late Payment Fee of 5% of the unpaid installment or $29 (whichever is less) is assessed. Payments that become 30 or more days past due are reported as delinquent to Equifax, Experian, and TransUnion, which can significantly lower your credit score.

--- Match 2 | Source: NorthStar_Bank_Credit_Card_Customer_Guide.pdf ---
### 19. Does NorthStar Bank have a late payment grace period?  
No. Unlike personal loans, credit cards do not have a post-due-date grace period for late fees. Payments received after 5:00 PM EST on your Payment Due Date are considered late and may incur a Late Payment Fee up to $30 (or $41 for repeat late payments within 6 cycles).

--- Match 3 | Source: NorthStar_Bank_Personal_Loan_Customer_Handbook.pdf ---
### Scenario 4: Navigating Temporary Fi

In [20]:
llm = ChatOpenAI(model=os.getenv("MODEL_NAME", "gpt-4o-mini"), temperature=0)

# ---- Cell B: route the question to the right guide, then retrieve ----
GUIDES = {
    "credit_card":     "NorthStar_Bank_Credit_Card_Customer_Guide.pdf",
    "personal_loan":   "NorthStar_Bank_Personal_Loan_Customer_Handbook.pdf",
    "digital_banking": "NorthStar_Bank_Digital_Banking_&_Account_Services_Guide.pdf",
}
router_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the banking question into exactly one label: "
               "credit_card, personal_loan, digital_banking, or all (if unclear or spans several). "
               "Reply with the label only."),
    ("human", "{question}"),
])
router = router_prompt | llm | StrOutputParser()

def retrieve(question: str, k: int = 5):
    label = router.invoke({"question": question}).strip().lower()
    search_kwargs = {"k": k}
    if label in GUIDES:
        search_kwargs["filter"] = {"source": GUIDES[label]}
    return vector_store.as_retriever(search_kwargs=search_kwargs).invoke(question)

# ---- Cell C: AUGMENTATION: turn retrieved chunks into numbered, cited context ----
def format_context(docs) -> str:
    blocks = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        heading = " > ".join(h for h in (m.get("section"), m.get("subsection")) if h)
        blocks.append(
            f"[{i}] Source: {m.get('source')} | Section: {heading or 'n/a'}"
            f"{' | contains TABLE' if m.get('has_table') else ''}\n{d.page_content.strip()}"
        )
    return "\n\n---\n\n".join(blocks)

SYSTEM_PROMPT = """You are NorthStar Bank's customer-support assistant.
Answer ONLY from the numbered context below. Rules:
- If the answer is not in the context, say: "I couldn't find that in NorthStar Bank's guides." Do not guess.
- Quote exact figures (fees, APRs, limits, phone numbers, timeframes) as written.
- Context may contain Markdown tables: read the correct row AND column (e.g. the specific card).
- If the question is ambiguous between products, state the answer for each relevant product.
- Cite sources inline as [1], [2] matching the context numbers.
- Be concise: a direct answer first, then brief supporting details.

Context:
{context}"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# ---- Cell D: chain: retrieve -> augment -> generate (keeps docs for citations) ----
rag_chain = (
    RunnablePassthrough.assign(docs=RunnableLambda(lambda x: retrieve(x["question"])))
    .assign(context=lambda x: format_context(x["docs"]))
    .assign(answer=rag_prompt | llm | StrOutputParser())
)

def ask(question: str):
    out = rag_chain.invoke({"question": question})
    print("Q:", question, "\nA:", out["answer"], "\nSources:")
    for i, d in enumerate(out["docs"], 1):
        print(f"  [{i}] {d.metadata['source']} | {d.metadata.get('section')}")
    return out

ask("What is the foreign transaction fee on the Travel Horizon card?")

# ---- optional: see exactly what the LLM receives (the augmented prompt) ----
docs = retrieve("How much is the late payment fee?")
print(rag_prompt.format(context=format_context(docs), question="How much is the late payment fee?"))


Q: What is the foreign transaction fee on the Travel Horizon card? 
A: The foreign transaction fee on the Travel Horizon card is $0 [3]. 
Sources:
  [1] NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 8. Frequently Asked Questions (52 Comprehensive FAQs)
  [2] NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 8. Frequently Asked Questions (52 Comprehensive FAQs)
  [3] NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 5. Fees, Rates & Charges Schedule
  [4] NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 4. Step-by-Step Processes
  [5] NorthStar_Bank_Credit_Card_Customer_Guide.pdf | 8. Frequently Asked Questions (52 Comprehensive FAQs)
System: You are NorthStar Bank's customer-support assistant.
Answer ONLY from the numbered context below. Rules:
- If the answer is not in the context, say: "I couldn't find that in NorthStar Bank's guides." Do not guess.
- Quote exact figures (fees, APRs, limits, phone numbers, timeframes) as written.
- Context may contain Markdown tables: read the correct r